# 02 — Clustering: Execução Individual de Algoritmos

**Objetivo**: executar cada algoritmo de clustering disponível no projeto
(K-Means, DBSCAN, Aglomerativo, Mean Shift) individualmente, avaliar os resultados
com métricas internas e visualizar os clusters formados.

## Tópicos
1. Pré-processamento
2. K-Means
3. DBSCAN
4. Clustering Aglomerativo
5. Mean Shift
6. Métricas internas
7. Visualização PCA 2D

In [ ]:
# Imports do projeto
from ml_clustering_lab.datasets.loaders import load_sklearn
from ml_clustering_lab.preprocessing.cleaning import drop_missing, drop_duplicates
from ml_clustering_lab.preprocessing.feature_selection import select_numeric_features
from ml_clustering_lab.preprocessing.scaling import scale_features
from ml_clustering_lab.clustering import get_algorithm
from ml_clustering_lab.clustering.evaluation import compute_internal_metrics
from ml_clustering_lab.visualization.embeddings import reduce_pca

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Pré-processamento

In [ ]:
df = load_sklearn('iris')

# Limpeza e seleção de features numéricas
df_clean = drop_missing(drop_duplicates(df))
X_df = select_numeric_features(df_clean, exclude=['target'])

# Escalonamento e conversão para ndarray
X_scaled = scale_features(X_df).values
print(f'Shape após pré-processamento: {X_scaled.shape}')

# Redução para visualização 2D
X_2d = reduce_pca(X_scaled)
print(f'Shape PCA 2D: {X_2d.shape}')

## 2. K-Means

In [ ]:
km = get_algorithm('kmeans', n_clusters=3)
labels_km = km.fit_predict(X_scaled)

metrics_km = compute_internal_metrics(X_scaled, labels_km)
print(f'K-Means — n_clusters={metrics_km["n_clusters"]}  '
      f'silhouette={metrics_km["silhouette"]:.4f}  '
      f'Davies-Bouldin={metrics_km["davies_bouldin"]:.4f}')

## 3. DBSCAN

In [ ]:
db = get_algorithm('dbscan', eps=0.5, min_samples=5)
labels_db = db.fit_predict(X_scaled)

metrics_db = compute_internal_metrics(X_scaled, labels_db)
print(f'DBSCAN — n_clusters={metrics_db["n_clusters"]}  '
      f'n_noise={metrics_db["n_noise"]}  '
      f'silhouette={metrics_db["silhouette"]:.4f}')

## 4. Clustering Aglomerativo

In [ ]:
agg = get_algorithm('agglomerative', n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)

metrics_agg = compute_internal_metrics(X_scaled, labels_agg)
print(f'Aglomerativo — n_clusters={metrics_agg["n_clusters"]}  '
      f'silhouette={metrics_agg["silhouette"]:.4f}  '
      f'Calinski-Harabász={metrics_agg["calinski_harabasz"]:.2f}')

## 5. Mean Shift

In [ ]:
ms = get_algorithm('mean-shift')
labels_ms = ms.fit_predict(X_scaled)

metrics_ms = compute_internal_metrics(X_scaled, labels_ms)
print(f'Mean Shift — n_clusters={metrics_ms["n_clusters"]}  '
      f'silhouette={metrics_ms["silhouette"]:.4f}  '
      f'Davies-Bouldin={metrics_ms["davies_bouldin"]:.4f}')

## 6. Comparação das métricas internas

In [ ]:
rows = [
    {'Algoritmo': 'K-Means',      **metrics_km},
    {'Algoritmo': 'DBSCAN',       **metrics_db},
    {'Algoritmo': 'Aglomerativo', **metrics_agg},
    {'Algoritmo': 'Mean Shift',   **metrics_ms},
]
cols = ['Algoritmo', 'n_clusters', 'n_noise', 'silhouette', 'davies_bouldin', 'calinski_harabasz']
pd.DataFrame(rows)[cols].set_index('Algoritmo').round(4)

## 7. Visualização em PCA 2D

In [ ]:
algo_labels = [
    ('K-Means',      labels_km),
    ('DBSCAN',       labels_db),
    ('Aglomerativo', labels_agg),
    ('Mean Shift',   labels_ms),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, labels) in zip(axes, algo_labels):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', s=30, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Clusters — PCA 2D — Iris', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Escolha do k ótimo

Quando não sabemos o número de clusters ideal, usamos dois métodos:

- **Método do Cotovelo (Elbow)**: plota a inércia vs k. O 'cotovelo' sugere o k ótimo.
- **Análise de Silhouette**: plota o Silhouette Score vs k. O pico indica o melhor k.


In [ ]:
from ml_clustering_lab.clustering.optimal_k import (
    elbow_analysis,
    silhouette_analysis,
    plot_elbow,
    plot_silhouette_analysis,
)

# Usamos X_scaled preparado na seção anterior
elbow_df = elbow_analysis(X_scaled, k_range=range(2, 10))
print(elbow_df)


In [ ]:
sil_df = silhouette_analysis(X_scaled, k_range=range(2, 10))
print(sil_df)

best_k = sil_df.loc[sil_df['silhouette'].idxmax(), 'k']
print(f'\nMelhor k pelo Silhouette Score: {int(best_k)}')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(elbow_df['k'], elbow_df['inertia'], marker='o', color='steelblue')
axes[0].set_title('Método do Cotovelo')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inércia (WCSS)')

axes[1].plot(sil_df['k'], sil_df['silhouette'], marker='o', color='darkorange')
axes[1].axvline(x=best_k, linestyle='--', color='red', alpha=0.6, label=f'k={int(best_k)}')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()
